## Integrantes
- Andre Marroquin
- Gabriel Paz
- Mauricio lemus

# Fase B Modelo Federado e Inferencia sobre Banco 3

Este cuaderno implementa el Objetivo B del proyecto de detección de fraude federada. Se usan tres datasets de transacciones de tarjeta basadas en ISO 8583. Banco 1 y Banco 2 tienen la variable objetivo `is_fraud` y se usan para entrenamiento y validación. Banco 3 se trata como no etiquetado.

El objetivo es entrenar una simulación federada razonable usando Banco 1 y Banco 2 y generar inferencias binarias para Banco 3. Se generan inferencias preliminares para el primer 30 por ciento y una inferencia final para el 100 por ciento.

El enfoque es inferencial. Primero se revisan columnas tipos valores faltantes y estructura antes de modelar.


## Environment setup

Se preparan librerías rutas constantes y carpetas de salida. El cuaderno requiere Python 3.12 o superior.


In [1]:
from pathlib import Path
import json
import sys
import warnings
from datetime import datetime

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import average_precision_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")
if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12 or higher is required")

random_seed = 42
np.random.seed(random_seed)

project_dir = Path.cwd()
data_dir = project_dir / "data"
output_dir = project_dir / "outputs_fase_B"
mini_eda_dir = output_dir / "01_mini_eda"
data_checks_dir = output_dir / "02_data_checks"
features_dir = output_dir / "03_features"
models_dir = output_dir / "04_models"
predictions_dir = output_dir / "05_predictions"
plots_dir = output_dir / "06_plots"
reports_dir = output_dir / "07_reports"

for folder in [output_dir, mini_eda_dir, data_checks_dir, features_dir, models_dir, predictions_dir, plots_dir, reports_dir]:
    folder.mkdir(parents=True, exist_ok=True)

bank_file_candidates = {
    "bank_1": ["Copia de 01_bo_vip_seed22_n100000.csv", "01_bo_vip_seed22_n100000.csv"],
    "bank_2": ["Copia de 02_br_privado_seed33_n100000.csv", "02_br_privado_seed33_n100000.csv"],
    "bank_3": ["Copia de 03_gt_estatal_seed3_n100000.csv", "03_gt_estatal_seed3_n100000.csv"],
}
target_column = "is_fraud"
preliminary_fraction = 0.30
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print(f"Project directory: {project_dir}")
print(f"Output directory: {output_dir}")
print(f"Python version: {sys.version.split()[0]}")


Project directory: d:\Proyecto-DataSience-PlusTI
Output directory: d:\Proyecto-DataSience-PlusTI\outputs_fase_B
Python version: 3.12.10


## Data loading helpers

Se definen funciones para localizar archivos en la raíz o en `data`, detectar separadores CSV, cargar datasets y normalizar nombres de columnas.

In [2]:
def normalize_to_ascii(value):
    return str(value).encode("ascii", errors="ignore").decode("ascii")


def to_snake_case(value):
    text = normalize_to_ascii(value).strip()
    text = pd.Series([text]).str.replace(r"[^0-9a-zA-Z]+", "_", regex=True).iloc[0]
    text = pd.Series([text]).str.replace(r"([a-z0-9])([A-Z])", r"\1_\2", regex=True).iloc[0]
    text = pd.Series([text]).str.replace(r"_+", "_", regex=True).iloc[0]
    return text.strip("_").lower() or "unnamed_column"


def normalize_column_names(columns):
    result = []
    seen = {}
    for column in columns:
        base = to_snake_case(column)
        count = seen.get(base, 0)
        result.append(base if count == 0 else f"{base}_{count + 1}")
        seen[base] = count + 1
    return result


def detect_csv_separator(path):
    sample = path.read_text(encoding="utf-8", errors="ignore")[:8192]
    counts = {sep: sample.count(sep) for sep in [";", ",", "\t", "|"]}
    best = max(counts, key=counts.get)
    return best if counts[best] > 0 else ","


def find_input_file(file_candidates):
    for folder in [project_dir, data_dir]:
        for file_name in file_candidates:
            path = folder / file_name
            if path.exists():
                return path
    raise FileNotFoundError("Input file not found")


def load_bank_dataset(bank_key, file_candidates):
    path = find_input_file(file_candidates)
    separator = detect_csv_separator(path)
    raw_data = pd.read_csv(path, sep=separator, low_memory=False)
    original_columns = list(raw_data.columns)
    normalized_columns = normalize_column_names(original_columns)
    data = raw_data.copy()
    data.columns = normalized_columns
    mapping = pd.DataFrame({"bank_key": bank_key, "original_column": original_columns, "normalized_column": normalized_columns})
    return {"bank_key": bank_key, "path": path, "separator": separator, "data": data, "mapping": mapping}


loaded_banks = {bank_key: load_bank_dataset(bank_key, candidates) for bank_key, candidates in bank_file_candidates.items()}
bank_1_data = loaded_banks["bank_1"]["data"]
bank_2_data = loaded_banks["bank_2"]["data"]
bank_3_data = loaded_banks["bank_3"]["data"]

load_summary = pd.DataFrame([
    {"bank_key": key, "file_name": item["path"].name, "separator": item["separator"], "row_count": len(item["data"]), "column_count": item["data"].shape[1]}
    for key, item in loaded_banks.items()
])
display(load_summary)


,bank_key,file_name,separator,row_count,column_count
0,bank_1,Copia de 01_bo_vip_seed22_n100000.csv,;,100003,66
1,bank_2,Copia de 02_br_privado_seed33_n100000.csv,;,100000,66
2,bank_3,Copia de 03_gt_estatal_seed3_n100000.csv,;,100000,66


## EDA inicial por archivo

Se ejecuta un miniEDA breve por banco para conocer forma columnas tipos nulos cardinalidad y estado inicial del target. Proceso parecido y repertido del objetivo A.


In [3]:
def has_valid_target_values(series):
    values = series.dropna().astype(str).str.strip().str.lower()
    values = values[values != ""]
    if values.empty:
        return False
    return set(values.unique()).issubset({"true", "false", "1", "0", "yes", "no", "y", "n"})


def target_distribution(series):
    table = series.value_counts(dropna=False).rename("count").reset_index()
    table.columns = ["target_value", "count"]
    table["rate"] = table["count"] / max(len(series), 1)
    return table


def create_mini_eda(bank_key, loaded_bank, show_target):
    data = loaded_bank["data"]
    has_target = target_column in data.columns
    valid_target = has_valid_target_values(data[target_column]) if has_target else False
    summary = pd.DataFrame({
        "bank_key": bank_key,
        "file_name": loaded_bank["path"].name,
        "separator": loaded_bank["separator"],
        "column_name": data.columns,
        "dtype": data.dtypes.astype(str).values,
        "missing_rate": data.isna().mean().values,
        "cardinality": data.nunique(dropna=False).values,
        "has_target": has_target,
        "target_has_valid_values": valid_target,
    })
    print("=" * 90)
    print(f"Dataset: {bank_key}")
    print(f"Rows: {len(data)}")
    print(f"Columns: {data.shape[1]}")
    print(f"Separator: {repr(loaded_bank['separator'])}")
    print(f"Has is_fraud: {has_target}")
    print(f"Valid is_fraud: {valid_target}")
    display(data.head())
    display(pd.Series(data.columns.tolist(), name="column_name").to_frame())
    display(data.dtypes.astype(str).rename("dtype").to_frame())
    display(data.isna().mean().rename("missing_rate").to_frame())
    display(data.nunique(dropna=False).rename("cardinality").to_frame())
    if show_target and has_target:
        display(target_distribution(data[target_column]))
    summary.to_csv(mini_eda_dir / f"{bank_key}_mini_eda_summary.csv", index=False)
    loaded_bank["mapping"].to_csv(mini_eda_dir / f"column_mapping_{bank_key}.csv", index=False)
    return summary


mini_eda_tables = [
    create_mini_eda("bank_1", loaded_banks["bank_1"], True),
    create_mini_eda("bank_2", loaded_banks["bank_2"], True),
    create_mini_eda("bank_3", loaded_banks["bank_3"], False),
]
load_summary["has_is_fraud"] = [target_column in loaded_banks[key]["data"].columns for key in load_summary["bank_key"]]
load_summary["target_has_valid_values"] = [has_valid_target_values(loaded_banks[key]["data"][target_column]) if target_column in loaded_banks[key]["data"].columns else False for key in load_summary["bank_key"]]
load_summary.to_csv(mini_eda_dir / "load_summary.csv", index=False)
display(load_summary)


Dataset: bank_1
Rows: 100003
Columns: 66
Separator: ';'
Has is_fraud: True
Valid is_fraud: True


,transaction_id,bank_code,bank_name,bank_country,bank_tier,client_id,client_segment,channel,card_brand,pan_masked,...,amount_usd,is_international,distance_from_home_km,hour_local,day_of_week,approved,response_description,client_baseline_amount,client_home_city,is_fraud
0,7dd812b1-bd03-4d05-afc6-c318dcc9b651,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00001325,PLATINUM,POS,MASTERCARD,531270******3773,...,500.12,True,8717.0,20,Tue,True,Approved,2012.51,TARIJA,False
1,c08b49a6-889a-491a-a1f8-974526f7886d,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00000079,PRIVATE,ECOM,VISA,421250******5552,...,1898.93,False,4.9,20,Tue,True,Approved,1096.46,LAPAZ,False
2,b04f88bd-2e33-42e5-a3cf-d52ef22dd7d9,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00002344,INFINITE,ECOM,NaN,531270******6104,...,349.85,False,4.4,20,Tue,True,Approved,1528.37,SANTACRUZ,False
3,3a836c25-7a8c-473b-8141-3e84ba3f212d,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00002587,PLATINUM,ATM,VISA,479500******0288,...,345.58,True,3966.0,20,Tue,True,Approved,2483.34,SUCRE,False
4,be9956da-924f-4c68-aed8-f0c5d949e577,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00000087,PRIVATE,POS,VISA,479500******0249,...,118.90,False,348.0,20,Tue,True,NaN,1334.55,SUCRE,False


,column_name
0,transaction_id
1,bank_code
2,bank_name
3,bank_country
4,bank_tier
...,...
61,approved
62,response_description
63,client_baseline_amount
64,client_home_city


,dtype
transaction_id,object
bank_code,object
bank_name,object
bank_country,object
bank_tier,object
...,...
approved,bool
response_description,object
client_baseline_amount,float64
client_home_city,object


,missing_rate
transaction_id,0.00000
bank_code,0.00000
bank_name,0.00000
bank_country,0.00000
bank_tier,0.00000
...,...
approved,0.00000
response_description,0.00988
client_baseline_amount,0.00000
client_home_city,0.00000


...

,column_name
0,transaction_id
1,bank_code
2,bank_name
3,bank_country
4,bank_tier
...,...
61,approved
62,response_description
63,client_baseline_amount
64,client_home_city


,dtype
transaction_id,object
bank_code,object
bank_name,object
bank_country,object
bank_tier,object
...,...
approved,bool
response_description,object
client_baseline_amount,float64
client_home_city,object


,missing_rate
transaction_id,0.00000
bank_code,0.00000
bank_name,0.00000
bank_country,0.00000
bank_tier,0.00000
...,...
approved,0.00000
response_description,0.02995
client_baseline_amount,0.00000
client_home_city,0.00000


,cardinality
transaction_id,100000
bank_code,1
bank_name,1
bank_country,1
bank_tier,1
...,...
approved,2
response_description,13
client_baseline_amount,4503
client_home_city,8


,bank_key,file_name,separator,row_count,column_count,has_is_fraud,target_has_valid_values
0,bank_1,Copia de 01_bo_vip_seed22_n100000.csv,;,100003,66,True,True
1,bank_2,Copia de 02_br_privado_seed33_n100000.csv,;,100000,66,True,True
2,bank_3,Copia de 03_gt_estatal_seed3_n100000.csv,;,100000,66,True,False


## Validación de variable objetivo

Banco 1 y Banco 2 deben tener `is_fraud` válido. Banco 3 se trata como no etiquetado.

In [4]:
def convert_target_to_binary(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)
    mapping = {"true": 1, "false": 0, "1": 1, "0": 0, "yes": 1, "no": 0, "y": 1, "n": 0}
    converted = series.astype(str).str.strip().str.lower().map(mapping)
    if converted.isna().any():
        bad_values = series[converted.isna()].dropna().astype(str).unique().tolist()[:20]
        raise ValueError(f"Unsupported target values: {bad_values}")
    return converted.astype(int)


def validate_labeled_target(bank_key, data):
    if target_column not in data.columns:
        raise ValueError(f"{bank_key} missing target")
    if not has_valid_target_values(data[target_column]):
        raise ValueError(f"{bank_key} target is invalid")
    target = convert_target_to_binary(data[target_column])
    if sorted(target.unique().tolist()) != [0, 1]:
        raise ValueError(f"{bank_key} target must contain 0 and 1")
    return target


bank_1_target = validate_labeled_target("bank_1", bank_1_data)
bank_2_target = validate_labeled_target("bank_2", bank_2_data)
bank_1_model_data = bank_1_data.copy()
bank_2_model_data = bank_2_data.copy()
bank_3_model_data = bank_3_data.copy()
bank_1_model_data[target_column] = bank_1_target
bank_2_model_data[target_column] = bank_2_target
bank_3_unlabeled_data = bank_3_model_data.drop(columns=[target_column], errors="ignore").copy()

target_validation_summary = pd.DataFrame([
    {"bank_key": "bank_1", "has_target": True, "target_used_for_training": True, "positive_count": int(bank_1_target.sum()), "negative_count": int((1 - bank_1_target).sum())},
    {"bank_key": "bank_2", "has_target": True, "target_used_for_training": True, "positive_count": int(bank_2_target.sum()), "negative_count": int((1 - bank_2_target).sum())},
    {"bank_key": "bank_3", "has_target": target_column in bank_3_data.columns, "target_used_for_training": False, "positive_count": None, "negative_count": None},
])
target_validation_summary.to_csv(data_checks_dir / "target_validation_summary.csv", index=False)
display(target_validation_summary)


,bank_key,has_target,target_used_for_training,positive_count,negative_count
0,bank_1,True,True,4919.0,95084.0
1,bank_2,True,True,3205.0,96795.0
2,bank_3,True,False,NaN,NaN


## Detección inferencial de columnas relevantes

Se detectan columnas por grupos semánticos usando reglas por nombres. Esta detección se basa en candidatos.

In [5]:
def find_columns_by_keywords(columns, keywords):
    return [column for column in columns if any(keyword in column.lower() for keyword in keywords)]


def choose_primary_column(candidates, preferred_columns):
    for column in preferred_columns:
        if column in candidates:
            return column
    return candidates[0] if candidates else None


def detect_semantic_columns(dataframes):
    all_columns = sorted(set().union(*[set(data.columns) for data in dataframes]))
    rules = {
        "transaction_id": (["transaction_id", "txn_id", "stan", "retrieval", "reference", "rrn"], ["transaction_id", "de11_stan", "de37_retrieval_reference_number"]),
        "date": (["datetime", "date", "transmission", "local_date", "settlement", "de7", "de13"], ["de7_transmission_datetime", "transaction_datetime", "de13_local_date"]),
        "amount": (["amount", "de4", "de6", "amt"], ["amount_usd", "amount_local", "amount_tx_currency", "de4_amount_transaction"]),
        "customer": (["client", "customer", "cardholder", "account_id", "de102"], ["client_id", "customer_id", "de102_account_id_1"]),
        "card": (["card", "pan", "hash", "brand", "de2"], ["pan_hash", "pan_masked", "card_brand", "de2_pan"]),
        "merchant": (["merchant", "acceptor", "de42", "de43"], ["de42_card_acceptor_id", "de43_card_acceptor_name_location"]),
        "mcc": (["mcc", "merchant_category", "de18"], ["de18_merchant_category_code"]),
        "country": (["country", "acquirer_country", "merchant_country", "cardholder_country"], ["bank_country", "de19_acquirer_country_code"]),
        "currency": (["currency", "de49", "de50", "de51"], ["currency_tx_alpha", "de49_currency_code_transaction"]),
        "channel": (["channel", "terminal_type", "de60"], ["channel", "de60_pos_terminal_type"]),
        "pos_entry_mode": (["pos_entry", "entry_mode", "de22", "pos_data", "de123"], ["de22_pos_entry_mode", "de123_pos_data_code"]),
        "device": (["device", "terminal", "wallet", "mobile", "de41", "de60"], ["de41_terminal_id", "de60_pos_terminal_type"]),
        "distance": (["distance", "home_km", "km"], ["distance_from_home_km"]),
        "hour": (["hour", "local_time", "de12"], ["hour_local", "de12_local_time"]),
        "approved_response": (["approved", "response", "de39"], ["approved", "de39_response_code", "response_description"]),
        "baseline_amount": (["baseline", "client_baseline", "normal_amount"], ["client_baseline_amount"]),
    }
    rows = []
    detected = {}
    for group, (keywords, preferred) in rules.items():
        candidates = find_columns_by_keywords(all_columns, keywords)
        primary = choose_primary_column(candidates, preferred)
        detected[group] = {"primary_column": primary, "candidate_columns": candidates}
        rows.append({"semantic_group": group, "primary_column": primary, "candidate_columns": " | ".join(candidates)})
    return detected, pd.DataFrame(rows)


detected_columns, detected_columns_table = detect_semantic_columns([bank_1_model_data, bank_2_model_data, bank_3_unlabeled_data])
detected_columns_table.to_csv(data_checks_dir / "detected_columns.csv", index=False)
with open(data_checks_dir / "detected_columns.json", "w", encoding="utf-8") as file:
    json.dump(detected_columns, file, indent=2, ensure_ascii=True)
display(detected_columns_table)


,semantic_group,primary_column,candidate_columns
0,transaction_id,transaction_id,de11_stan | de37_retrieval_reference_number | ...
1,date,de7_transmission_datetime,de13_local_date | de14_expiration_date | de15_...
2,amount,amount_usd,amount_local | amount_tx_currency | amount_usd...
3,customer,client_id,client_baseline_amount | client_home_city | cl...
4,card,pan_hash,card_brand | de22_pos_entry_mode | de23_card_s...
5,merchant,de42_card_acceptor_id,de18_merchant_category_code | de42_card_accept...
6,mcc,de18_merchant_category_code,de18_merchant_category_code
7,country,bank_country,bank_country | de19_acquirer_country_code
8,currency,currency_tx_alpha,amount_tx_currency | currency_tx_alpha | de49_...
9,channel,channel,channel | de60_pos_terminal_type


## Ingeniería de variables

Se crean variables derivadas generales entre bancos. Cada variable se genera solo si existe información suficiente.

In [6]:
def get_primary_column(detected, group):
    return detected.get(group, {}).get("primary_column")


def to_numeric_safe(series):
    return pd.to_numeric(series, errors="coerce")


def normalize_text_values(series):
    return series.astype(str).str.upper().str.strip()


def parse_date_values(series):
    parsed = pd.to_datetime(series, errors="coerce")
    clean = series.astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
    mask = parsed.isna() & clean.str.len().ge(4)
    if mask.any():
        padded = clean[mask].str.zfill(10)
        frame = pd.DataFrame({
            "year": 2025,
            "month": pd.to_numeric(padded.str.slice(0, 2), errors="coerce"),
            "day": pd.to_numeric(padded.str.slice(2, 4), errors="coerce"),
            "hour": pd.to_numeric(padded.str.slice(4, 6), errors="coerce"),
            "minute": pd.to_numeric(padded.str.slice(6, 8), errors="coerce"),
            "second": pd.to_numeric(padded.str.slice(8, 10), errors="coerce"),
        })
        parsed.loc[mask] = pd.to_datetime(frame, errors="coerce")
    return parsed


def extract_hour_values(data, detected):
    hour_column = get_primary_column(detected, "hour")
    if hour_column in data.columns:
        if "hour" in hour_column:
            return to_numeric_safe(data[hour_column])
        clean = data[hour_column].astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True).str.zfill(6)
        return pd.to_numeric(clean.str.slice(0, 2), errors="coerce")
    date_column = get_primary_column(detected, "date")
    if date_column in data.columns:
        return parse_date_values(data[date_column]).dt.hour
    return pd.Series(np.nan, index=data.index)


def build_feature_frame(data, detected, bank_key):
    result = data.copy()
    rows = []
    amount_column = get_primary_column(detected, "amount")
    amount = to_numeric_safe(result[amount_column]) if amount_column in result.columns else pd.Series(np.nan, index=result.index)
    result["amount_numeric"] = amount
    result["log_amount"] = np.log1p(amount.clip(lower=0))
    baseline_column = get_primary_column(detected, "baseline_amount")
    baseline = to_numeric_safe(result[baseline_column]) if baseline_column in result.columns else pd.Series(np.nan, index=result.index)
    result["amount_to_baseline_ratio"] = amount / baseline.replace(0, np.nan)
    distance_column = get_primary_column(detected, "distance")
    distance = to_numeric_safe(result[distance_column]) if distance_column in result.columns else pd.Series(np.nan, index=result.index)
    result["distance_from_home_numeric"] = distance
    result["log_distance_from_home"] = np.log1p(distance.clip(lower=0))
    result["missing_distance_indicator"] = distance.isna().astype(int)
    hour = extract_hour_values(result, detected)
    result["hour"] = hour
    result["night_transaction_flag"] = hour.isin([0, 1, 2, 3, 4, 5]).astype(int)
    result["business_hour_transaction_flag"] = hour.between(9, 17).astype(int)
    date_column = get_primary_column(detected, "date")
    dates = parse_date_values(result[date_column]) if date_column in result.columns else pd.Series(pd.NaT, index=result.index)
    result["month"] = dates.dt.month
    result["day"] = dates.dt.day
    result["weekday"] = dates.dt.weekday
    channel_column = get_primary_column(detected, "channel")
    pos_column = get_primary_column(detected, "pos_entry_mode")
    device_column = get_primary_column(detected, "device")
    channel = normalize_text_values(result[channel_column]) if channel_column in result.columns else pd.Series("", index=result.index)
    pos = normalize_text_values(result[pos_column]) if pos_column in result.columns else pd.Series("", index=result.index)
    device = normalize_text_values(result[device_column]) if device_column in result.columns else pd.Series("", index=result.index)
    result["ecommerce_channel_proxy"] = (channel.str.contains("ECOM", na=False) | device.str.contains("ECOM|VIRTUAL|ONLINE", regex=True, na=False) | pos.isin(["81", "081"])).astype(int)
    result["manual_or_ecommerce_entry_proxy"] = (pos.isin(["01", "001", "81", "081"]) | pos.str.contains("MANUAL|ECOM", regex=True, na=False)).astype(int)
    result["contactless_proxy"] = (pos.str.contains("CONTACTLESS|071|07", regex=True, na=False) | device.str.contains("CONTACTLESS|NFC|WALLET", regex=True, na=False)).astype(int)
    result["international_flag"] = normalize_text_values(result["is_international"]).isin(["TRUE", "1", "YES", "Y"]).astype(int) if "is_international" in result.columns else np.nan
    response_column = get_primary_column(detected, "approved_response")
    response = normalize_text_values(result[response_column]) if response_column in result.columns else pd.Series("", index=result.index)
    result["approved_response_proxy"] = response.isin(["TRUE", "1", "0", "APPROVED"]).astype(int)
    for feature in ["amount_numeric", "log_amount", "amount_to_baseline_ratio", "distance_from_home_numeric", "log_distance_from_home", "missing_distance_indicator", "hour", "night_transaction_flag", "business_hour_transaction_flag", "month", "day", "weekday", "ecommerce_channel_proxy", "manual_or_ecommerce_entry_proxy", "contactless_proxy", "international_flag", "approved_response_proxy"]:
        rows.append({"bank_key": bank_key, "feature_name": feature, "status": "created"})
    return result, pd.DataFrame(rows)


bank_1_features, bank_1_feature_summary = build_feature_frame(bank_1_model_data, detected_columns, "bank_1")
bank_2_features, bank_2_feature_summary = build_feature_frame(bank_2_model_data, detected_columns, "bank_2")
bank_3_features, bank_3_feature_summary = build_feature_frame(bank_3_unlabeled_data, detected_columns, "bank_3")
feature_engineering_summary = pd.concat([bank_1_feature_summary, bank_2_feature_summary, bank_3_feature_summary], ignore_index=True)
feature_engineering_summary.to_csv(features_dir / "feature_engineering_summary.csv", index=False)
display(feature_engineering_summary)

,bank_key,feature_name,status
0,bank_1,amount_numeric,created
1,bank_1,log_amount,created
2,bank_1,amount_to_baseline_ratio,created
3,bank_1,distance_from_home_numeric,created
4,bank_1,log_distance_from_home,created
5,bank_1,missing_distance_indicator,created
6,bank_1,hour,created
7,bank_1,night_transaction_flag,created
8,bank_1,business_hour_transaction_flag,created
9,bank_1,month,created


## Control de fuga de información y selección de variables

Se usan columnas comunes entre los tres bancos y se eliminan target fraude identificadores banco país moneda ciudad y columnas poco generalizables.

In [7]:
def get_exclusion_reason(column):
    name = column.lower()
    rules = [
        ("target_or_fraud", ["is_fraud", "fraud", "scenario"]),
        ("bank_identifier", ["bank", "source_bank", "bank_tier"]),
        ("local_geography", ["country", "city", "home_city"]),
        ("local_currency", ["currency"]),
        ("direct_identifier", ["transaction_id", "customer_id", "client_id", "pan", "hash", "stan", "rrn", "retrieval", "terminal_id", "merchant_id", "acceptor_id", "account_id", "authorization", "track", "de35", "de37", "de38", "de41", "de42", "de102", "de103"]),
        ("raw_date_identifier", ["de7_transmission_datetime", "de12_local_time", "de13_local_date", "de14_expiration_date", "de15_settlement_date"]),
    ]
    for reason, patterns in rules:
        if any(pattern in name for pattern in patterns):
            return reason
    return None


def select_common_features(dataframes):
    common = sorted(set(dataframes[0].columns).intersection(*[set(data.columns) for data in dataframes[1:]]))
    reference = pd.concat([dataframes[0][common], dataframes[1][common]], ignore_index=True)
    selected = []
    excluded = []
    types = []
    for column in common:
        reason = get_exclusion_reason(column)
        missing = float(reference[column].isna().mean())
        cardinality = int(reference[column].nunique(dropna=False))
        unique_rate = cardinality / max(len(reference), 1)
        is_numeric = pd.api.types.is_numeric_dtype(reference[column])
        if reason is None and missing > 0.95:
            reason = "high_missing_rate"
        if reason is None and not is_numeric and (cardinality > 500 or unique_rate > 0.30):
            reason = "high_cardinality"
        if reason is None:
            selected.append(column)
            types.append({"feature_name": column, "feature_type": "numeric" if is_numeric else "categorical", "dtype": str(reference[column].dtype), "missing_rate": missing, "cardinality": cardinality})
        else:
            excluded.append({"feature_name": column, "reason": reason, "dtype": str(reference[column].dtype), "missing_rate": missing, "cardinality": cardinality})
    return selected, pd.DataFrame(excluded), pd.DataFrame(types)


selected_features, excluded_features_table, feature_type_summary = select_common_features([bank_1_features, bank_2_features, bank_3_features])
if target_column in selected_features:
    raise ValueError("Target cannot be part of selected features")
if not selected_features:
    raise ValueError("No selected features found")
numeric_features = feature_type_summary.loc[feature_type_summary["feature_type"] == "numeric", "feature_name"].tolist()
categorical_features = feature_type_summary.loc[feature_type_summary["feature_type"] == "categorical", "feature_name"].tolist()
feature_selection_summary = pd.DataFrame([
    {"metric": "selected_feature_count", "value": len(selected_features)},
    {"metric": "numeric_feature_count", "value": len(numeric_features)},
    {"metric": "categorical_feature_count", "value": len(categorical_features)},
    {"metric": "excluded_feature_count", "value": len(excluded_features_table)},
])
with open(features_dir / "selected_features.json", "w", encoding="utf-8") as file:
    json.dump({"selected_features": selected_features}, file, indent=2, ensure_ascii=True)
with open(features_dir / "excluded_features.json", "w", encoding="utf-8") as file:
    json.dump(excluded_features_table.to_dict(orient="records"), file, indent=2, ensure_ascii=True)
feature_type_summary.to_csv(features_dir / "feature_type_summary.csv", index=False)
feature_selection_summary.to_csv(features_dir / "feature_selection_summary.csv", index=False)
display(feature_selection_summary)
display(feature_type_summary)
display(excluded_features_table.head(30))

,metric,value
0,selected_feature_count,44
1,numeric_feature_count,34
2,categorical_feature_count,10
3,excluded_feature_count,38


,feature_name,feature_type,dtype,missing_rate,cardinality
0,amount_local,numeric,float64,0.000000,138951
1,amount_numeric,numeric,float64,0.000000,66393
2,amount_to_baseline_ratio,numeric,float64,0.000000,199906
3,amount_usd,numeric,float64,0.000000,66393
4,approved,numeric,bool,0.000000,2
5,approved_response_proxy,numeric,int32,0.000000,2
6,business_hour_transaction_flag,numeric,int32,0.000000,2
7,card_brand,categorical,object,0.010260,3
8,channel,categorical,object,0.000000,4
9,client_baseline_amount,numeric,float64,0.000000,7796


,feature_name,reason,dtype,missing_rate,cardinality
0,amount_tx_currency,local_currency,float64,0.000000,132434
1,bank_code,bank_identifier,object,0.000000,2
2,bank_country,bank_identifier,object,0.000000,2
3,bank_name,bank_identifier,object,0.000000,2
4,bank_tier,bank_identifier,object,0.000000,2
5,client_home_city,local_geography,object,0.000000,16
6,client_id,direct_identifier,object,0.000000,8000
7,currency_tx_alpha,local_currency,object,0.000000,7
8,de102_account_id_1,direct_identifier,object,0.010080,8001
9,de103_account_id_2,direct_identifier,float64,1.000000,1


## Separación de entrenamiento y validación

Se crean splits estratificados independientes para Banco 1 y Banco 2. Luego se arma un conjunto centralizado con los datos de entrenamiento de ambos bancos.

In [8]:

def create_bank_split(features, target, bank_key):
    x_data = features[selected_features].copy()
    train_index, valid_index = train_test_split(x_data.index, test_size=0.25, random_state=random_seed, stratify=target)
    return {"bank_key": bank_key, "x_train": x_data.loc[train_index], "x_valid": x_data.loc[valid_index], "y_train": target.loc[train_index], "y_valid": target.loc[valid_index]}


bank_1_split = create_bank_split(bank_1_features, bank_1_target, "bank_1")
bank_2_split = create_bank_split(bank_2_features, bank_2_target, "bank_2")
central_x_train = pd.concat([bank_1_split["x_train"], bank_2_split["x_train"]], ignore_index=True)
central_y_train = pd.concat([bank_1_split["y_train"], bank_2_split["y_train"]], ignore_index=True)
central_x_valid = pd.concat([bank_1_split["x_valid"], bank_2_split["x_valid"]], ignore_index=True)
central_y_valid = pd.concat([bank_1_split["y_valid"], bank_2_split["y_valid"]], ignore_index=True)
validation_splits = {"bank_1_valid": (bank_1_split["x_valid"], bank_1_split["y_valid"]), "bank_2_valid": (bank_2_split["x_valid"], bank_2_split["y_valid"]), "central_valid": (central_x_valid, central_y_valid)}
train_validation_summary = pd.DataFrame([
    {"split_name": "bank_1_train", "rows": len(bank_1_split["x_train"]), "positive_rate": float(bank_1_split["y_train"].mean())},
    {"split_name": "bank_1_valid", "rows": len(bank_1_split["x_valid"]), "positive_rate": float(bank_1_split["y_valid"].mean())},
    {"split_name": "bank_2_train", "rows": len(bank_2_split["x_train"]), "positive_rate": float(bank_2_split["y_train"].mean())},
    {"split_name": "bank_2_valid", "rows": len(bank_2_split["x_valid"]), "positive_rate": float(bank_2_split["y_valid"].mean())},
    {"split_name": "central_train", "rows": len(central_x_train), "positive_rate": float(central_y_train.mean())},
    {"split_name": "central_valid", "rows": len(central_x_valid), "positive_rate": float(central_y_valid.mean())},
])
train_validation_summary.to_csv(data_checks_dir / "train_validation_summary.csv", index=False)
display(train_validation_summary)


,split_name,rows,positive_rate
0,bank_1_train,75002,0.049185
1,bank_1_valid,25001,0.049198
2,bank_2_train,75000,0.032053
3,bank_2_valid,25000,0.032040
4,central_train,150002,0.040619
5,central_valid,50001,0.040619


## Preprocesamiento

Se usa `ColumnTransformer` con imputación de mediana para numéricas e imputación de valor frecuente más OneHotEncoder para categóricas.

In [9]:
def create_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def build_preprocessor(scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    transformers = []
    if numeric_features:
        transformers.append(("numeric", Pipeline(numeric_steps), numeric_features))
    if categorical_features:
        transformers.append(("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", create_one_hot_encoder())]), categorical_features))
    if not transformers:
        raise ValueError("No features for preprocessing")
    return ColumnTransformer(transformers=transformers, remainder="drop")


def build_ordinal_preprocessor():
    transformers = []
    if numeric_features:
        transformers.append(("numeric", SimpleImputer(strategy="median"), numeric_features))
    if categorical_features:
        categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))])
        transformers.append(("categorical", categorical_pipeline, categorical_features))
    if not transformers:
        raise ValueError("No features for preprocessing")
    return ColumnTransformer(transformers=transformers, remainder="drop")


print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

Numeric features: 34
Categorical features: 10


## Estrategia federada simulada

No se implementa federated learning distribuido real. Se simula un escenario federado entrenando modelos locales por banco. Luego se combinan probabilidades de modelos locales y se comparan contra un baseline centralizado.

In [10]:

def build_extra_trees_pipeline():
    model = ExtraTreesClassifier(n_estimators=120, max_depth=16, min_samples_leaf=20, class_weight="balanced", random_state=random_seed, n_jobs=-1)
    return Pipeline([("preprocessor", build_preprocessor(False)), ("model", model)])


def build_logistic_pipeline():
    model = LogisticRegression(max_iter=500, class_weight="balanced", random_state=random_seed, n_jobs=-1)
    return Pipeline([("preprocessor", build_preprocessor(True)), ("model", model)])


def build_hist_gradient_boosting_pipeline():
    try:
        model = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=0.05, class_weight="balanced", random_state=random_seed)
    except TypeError:
        model = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=0.05, random_state=random_seed)
    return Pipeline([("preprocessor", build_ordinal_preprocessor()), ("model", model)])


def fit_and_save_model(strategy_name, pipeline, x_train, y_train):
    pipeline.fit(x_train, y_train)
    joblib.dump(pipeline, models_dir / f"{strategy_name}.joblib")
    return pipeline


bank_1_local_model = fit_and_save_model("bank_1_local_model", build_extra_trees_pipeline(), bank_1_split["x_train"], bank_1_split["y_train"])
bank_2_local_model = fit_and_save_model("bank_2_local_model", build_extra_trees_pipeline(), bank_2_split["x_train"], bank_2_split["y_train"])
centralized_model = fit_and_save_model("baseline_centralized_model", build_extra_trees_pipeline(), central_x_train, central_y_train)
central_logistic_model = fit_and_save_model("central_logistic_baseline", build_logistic_pipeline(), central_x_train, central_y_train)
central_hist_gradient_boosting_model = fit_and_save_model("central_hist_gradient_boosting_model", build_hist_gradient_boosting_pipeline(), central_x_train, central_y_train)
model_registry = {"bank_1_local_model": bank_1_local_model, "bank_2_local_model": bank_2_local_model, "baseline_centralized_model": centralized_model, "central_logistic_baseline": central_logistic_model, "central_hist_gradient_boosting_model": central_hist_gradient_boosting_model}
print("Models trained")

Models trained


## Métricas y selección de umbral

Se calculan accuracy precision recall F1 ROC AUC average precision matriz de confusión false positive ratio y false negative ratio. Se prueban umbrales de 0.10 a 0.90.

In [11]:
def predict_model_probability(model, x_data):
    return model.predict_proba(x_data)[:, 1]


def safe_roc_auc(y_true, y_score):
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)


def safe_average_precision(y_true, y_score):
    return np.nan if len(np.unique(y_true)) < 2 else average_precision_score(y_true, y_score)


def compute_metrics(y_true, y_score, threshold):
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    alert_count = tp + fp
    positive_count = tp + fn
    return {"threshold": float(threshold), "accuracy": accuracy_score(y_true, y_pred), "precision": precision_score(y_true, y_pred, zero_division=0), "recall": recall_score(y_true, y_pred, zero_division=0), "f1": f1_score(y_true, y_pred, zero_division=0), "roc_auc": safe_roc_auc(y_true, y_score), "average_precision": safe_average_precision(y_true, y_score), "true_negatives": int(tn), "false_positives": int(fp), "false_negatives": int(fn), "true_positives": int(tp), "false_positive_ratio": float(fp / alert_count) if alert_count else 0.0, "false_negative_ratio": float(fn / positive_count) if positive_count else 0.0}


def compute_local_weights():
    rows = []
    for model_name in ["bank_1_local_model", "bank_2_local_model"]:
        model = model_registry[model_name]
        scores = []
        for split_name in ["bank_1_valid", "bank_2_valid"]:
            x_valid, y_valid = validation_splits[split_name]
            scores.append(safe_roc_auc(y_valid, predict_model_probability(model, x_valid)))
        score = float(np.nanmean(scores)) if scores else 0.5
        rows.append({"model_name": model_name, "weight_score": max(score, 0.001)})
    table = pd.DataFrame(rows)
    table["weight"] = table["weight_score"] / table["weight_score"].sum()
    return table


local_weight_table = compute_local_weights()
performance_weights = dict(zip(local_weight_table["model_name"], local_weight_table["weight"]))
equal_weights = {"bank_1_local_model": 0.5, "bank_2_local_model": 0.5}
local_weight_table.to_csv(models_dir / "federated_model_weights.csv", index=False)
display(local_weight_table)


def predict_strategy_probability(strategy_name, x_data):
    if strategy_name in model_registry:
        return predict_model_probability(model_registry[strategy_name], x_data)
    if strategy_name == "federated_equal_weight_model":
        return sum(predict_model_probability(model_registry[name], x_data) * weight for name, weight in equal_weights.items())
    if strategy_name == "federated_performance_weighted_model":
        return sum(predict_model_probability(model_registry[name], x_data) * weight for name, weight in performance_weights.items())
    raise ValueError(f"Unknown strategy: {strategy_name}")


def evaluate_strategy_on_split(strategy_name, split_name, x_valid, y_valid, thresholds):
    y_score = predict_strategy_probability(strategy_name, x_valid)
    rows = []
    for threshold in thresholds:
        row = compute_metrics(y_valid, y_score, threshold)
        row.update({"strategy_name": strategy_name, "split_name": split_name})
        rows.append(row)
    return pd.DataFrame(rows)


threshold_grid = np.round(np.arange(0.10, 0.91, 0.05), 2)
strategy_names = ["baseline_centralized_model", "federated_equal_weight_model", "federated_performance_weighted_model", "central_logistic_baseline", "central_hist_gradient_boosting_model"]
threshold_metrics_table = pd.concat([evaluate_strategy_on_split(strategy, split, x_valid, y_valid, threshold_grid) for strategy in strategy_names for split, (x_valid, y_valid) in validation_splits.items()], ignore_index=True)
selected_rows = []
for strategy in strategy_names:
    central_rows = threshold_metrics_table[(threshold_metrics_table["strategy_name"] == strategy) & (threshold_metrics_table["split_name"] == "central_valid")].copy()
    f1_row = central_rows.sort_values(["f1", "recall", "precision"], ascending=[False, False, False]).iloc[0].to_dict()
    f1_row["threshold_type"] = "best_f1"
    recall_rows = central_rows[central_rows["recall"] >= 0.80].copy()
    if recall_rows.empty:
        recall_rows = central_rows.copy()
    recall_row = recall_rows.sort_values(["recall", "f1", "false_positive_ratio"], ascending=[False, False, True]).iloc[0].to_dict()
    recall_row["threshold_type"] = "recall_priority"
    selected_rows.extend([f1_row, recall_row])
selected_thresholds_table = pd.DataFrame(selected_rows)
validation_metrics_table = threshold_metrics_table.merge(selected_thresholds_table[["strategy_name", "threshold_type", "threshold"]], on=["strategy_name", "threshold"], how="inner")
cross_bank_validation_table = pd.DataFrame([
    dict(compute_metrics(bank_2_split["y_valid"], predict_strategy_probability("bank_1_local_model", bank_2_split["x_valid"]), 0.50), strategy_name="bank_1_local_model", split_name="cross_bank_bank_1_to_bank_2", threshold_type="fixed_0_50"),
    dict(compute_metrics(bank_1_split["y_valid"], predict_strategy_probability("bank_2_local_model", bank_1_split["x_valid"]), 0.50), strategy_name="bank_2_local_model", split_name="cross_bank_bank_2_to_bank_1", threshold_type="fixed_0_50"),
])
threshold_metrics_table.to_csv(reports_dir / "threshold_metrics.csv", index=False)
selected_thresholds_table.to_csv(reports_dir / "selected_thresholds.csv", index=False)
validation_metrics_table.to_csv(reports_dir / "model_validation_metrics.csv", index=False)
cross_bank_validation_table.to_csv(reports_dir / "cross_bank_validation_metrics.csv", index=False)
display(selected_thresholds_table)
display(validation_metrics_table)
display(cross_bank_validation_table)

,model_name,weight_score,weight
0,bank_1_local_model,0.862692,0.529012
1,bank_2_local_model,0.768070,0.470988


,threshold,accuracy,precision,recall,f1,roc_auc,average_precision,true_negatives,false_positives,false_negatives,true_positives,false_positive_ratio,false_negative_ratio,strategy_name,split_name,threshold_type
0,0.80,0.976700,0.790214,0.580502,0.669316,0.876442,0.681003,47657,313,852,1179,0.209786,0.419498,baseline_centralized_model,central_valid,best_f1
1,0.10,0.084238,0.042089,0.990153,0.080745,0.876442,0.681003,2201,45769,20,2011,0.957911,0.009847,baseline_centralized_model,central_valid,recall_priority
2,0.45,0.945341,0.396582,0.662728,0.496221,0.863972,0.527016,45922,2048,685,1346,0.603418,0.337272,federated_equal_weight_model,central_valid,best_f1
3,0.10,0.061319,0.041234,0.993599,0.079182,0.863972,0.527016,1048,46922,13,2018,0.958766,0.006401,federated_equal_weight_model,central_valid,recall_priority
4,0.45,0.945081,0.397122,0.679468,0.501271,0.864759,0.532932,45875,2095,651,1380,0.602878,0.320532,federated_performance_weighted_model,central_valid,best_f1
5,0.10,0.062399,0.041261,0.993107,0.079230,0.864759,0.532932,1103,46867,14,2017,0.958739,0.006893,federated_performance_weighted_model,central_valid,recall_priority
6,0.90,0.973461,0.829588,0.436238,0.571797,0.863365,0.584405,47788,182,1145,886,0.170412,0.563762,central_logistic_baseline,central_valid,best_f1
7,0.10,0.075998,0.041805,0.992122,0.080229,0.863365,0.584405,1785,46185,16,2015,0.958195,0.007878,central_logistic_baseline,central_valid,recall_priority
8,0.85,0.986360,0.963574,0.690300,0.804360,0.881434,0.758921,47917,53,629,1402,0.036426,0.309700,central_hist_gradient_boosting_model,central_valid,best_f1
9,0.10,0.069319,0.041631,0.995076,0.079918,0.881434,0.758921,1445,46525,10,2021,0.958369,0.004924,central_hist_gradient_boosting_model,central_valid,recall_priority


,threshold,accuracy,precision,recall,f1,roc_auc,average_precision,true_negatives,false_positives,false_negatives,true_positives,false_positive_ratio,false_negative_ratio,strategy_name,split_name,threshold_type
0,0.10,0.088476,0.050761,0.990244,0.096571,0.874479,0.635727,994,22777,12,1218,0.949239,0.009756,baseline_centralized_model,bank_1_valid,recall_priority
1,0.80,0.969001,0.792041,0.501626,0.614236,0.874479,0.635727,23609,162,613,617,0.207959,0.498374,baseline_centralized_model,bank_1_valid,best_f1
2,0.10,0.080000,0.033340,0.990012,0.064508,0.861784,0.723092,1207,22992,8,793,0.966660,0.009988,baseline_centralized_model,bank_2_valid,recall_priority
3,0.80,0.984400,0.788219,0.701623,0.742404,0.861784,0.723092,24048,151,239,562,0.211781,0.298377,baseline_centralized_model,bank_2_valid,best_f1
4,0.10,0.084238,0.042089,0.990153,0.080745,0.876442,0.681003,2201,45769,20,2011,0.957911,0.009847,baseline_centralized_model,central_valid,recall_priority
5,0.80,0.976700,0.790214,0.580502,0.669316,0.876442,0.681003,47657,313,852,1179,0.209786,0.419498,baseline_centralized_model,central_valid,best_f1
6,0.10,0.064197,0.049691,0.994309,0.094652,0.851497,0.385831,382,23389,7,1223,0.950309,0.005691,federated_equal_weight_model,bank_1_valid,recall_priority
7,0.45,0.914043,0.313287,0.626829,0.417773,0.851497,0.385831,22081,1690,459,771,0.686713,0.373171,federated_equal_weight_model,bank_1_valid,best_f1
8,0.10,0.058440,0.032678,0.992509,0.063274,0.858926,0.686751,666,23533,6,795,0.967322,0.007491,federated_equal_weight_model,bank_2_valid,recall_priority
9,0.45,0.976640,0.616292,0.717853,0.663206,0.858926,0.686751,23841,358,226,575,0.383708,0.282147,federated_equal_weight_model,bank_2_valid,best_f1


,threshold,accuracy,precision,recall,f1,roc_auc,average_precision,true_negatives,false_positives,false_negatives,true_positives,false_positive_ratio,false_negative_ratio,strategy_name,split_name,threshold_type
0,0.5,0.958680,0.403974,0.609238,0.485814,0.850729,0.536120,23479,720,313,488,0.596026,0.390762,bank_1_local_model,cross_bank_bank_1_to_bank_2,fixed_0_50
1,0.5,0.902564,0.187565,0.294309,0.229114,0.676717,0.165822,22203,1568,868,362,0.812435,0.705691,bank_2_local_model,cross_bank_bank_2_to_bank_1,fixed_0_50


## Selección de metodología final

La estrategia final se selecciona automáticamente con F1 promedio F1 mínimo entre bancos ROC AUC promedio recall y penalización por false positive ratio.

In [12]:
def get_selected_threshold(strategy_name, threshold_type="best_f1"):
    rows = selected_thresholds_table[(selected_thresholds_table["strategy_name"] == strategy_name) & (selected_thresholds_table["threshold_type"] == threshold_type)]
    return 0.50 if rows.empty else float(rows.iloc[0]["threshold"])


def build_selection_table(candidate_strategies):
    rows = []
    for strategy in candidate_strategies:
        threshold = get_selected_threshold(strategy)
        split_rows = []
        for split in ["bank_1_valid", "bank_2_valid", "central_valid"]:
            x_valid, y_valid = validation_splits[split]
            row = compute_metrics(y_valid, predict_strategy_probability(strategy, x_valid), threshold)
            row["split_name"] = split
            split_rows.append(row)
        table = pd.DataFrame(split_rows)
        mean_f1 = float(table["f1"].mean())
        min_f1 = float(table.loc[table["split_name"].isin(["bank_1_valid", "bank_2_valid"]), "f1"].min())
        mean_roc = float(table["roc_auc"].mean())
        mean_recall = float(table["recall"].mean())
        mean_fpratio = float(table["false_positive_ratio"].mean())
        internal_score = 0.30 * mean_f1 + 0.25 * min_f1 + 0.20 * mean_roc + 0.15 * mean_recall - 0.10 * mean_fpratio
        rows.append({"strategy_name": strategy, "threshold": threshold, "mean_f1": mean_f1, "min_f1_between_banks": min_f1, "mean_roc_auc": mean_roc, "mean_recall": mean_recall, "mean_false_positive_ratio": mean_fpratio, "internal_score": internal_score})
    return pd.DataFrame(rows)


selection_table = build_selection_table(strategy_names)
feedback_path = project_dir / "optional_external_feedback.csv"
if not feedback_path.exists() and data_dir.exists():
    feedback_path = data_dir / "optional_external_feedback.csv"
if feedback_path.exists():
    feedback = pd.read_csv(feedback_path)
    if {"strategy_name", "feedback_score"}.issubset(feedback.columns):
        selection_table = selection_table.merge(feedback[["strategy_name", "feedback_score"]], on="strategy_name", how="left")
        selection_table["feedback_score"] = selection_table["feedback_score"].fillna(selection_table["feedback_score"].median())
        max_feedback = selection_table["feedback_score"].max()
        selection_table["normalized_feedback_score"] = selection_table["feedback_score"] / max_feedback if max_feedback > 0 else 0.0
        selection_table["final_score"] = 0.80 * selection_table["internal_score"] + 0.20 * selection_table["normalized_feedback_score"]
    else:
        selection_table["final_score"] = selection_table["internal_score"]
else:
    selection_table["final_score"] = selection_table["internal_score"]
selection_table = selection_table.sort_values("final_score", ascending=False).reset_index(drop=True)
selected_strategy_name = selection_table.iloc[0]["strategy_name"]
selected_strategy_threshold = float(selection_table.iloc[0]["threshold"])
selection_summary = {"selected_strategy_name": selected_strategy_name, "selected_strategy_threshold": selected_strategy_threshold, "external_feedback_used": bool(feedback_path.exists()), "created_at": datetime.now().isoformat(timespec="seconds")}
selection_table.to_csv(reports_dir / "selection_table.csv", index=False)
with open(reports_dir / "selection_summary.json", "w", encoding="utf-8") as file:
    json.dump(selection_summary, file, indent=2, ensure_ascii=True)
display(selection_table)
print(f"Selected strategy: {selected_strategy_name}")


,strategy_name,threshold,mean_f1,min_f1_between_banks,mean_roc_auc,mean_recall,mean_false_positive_ratio,internal_score,final_score
0,central_hist_gradient_boosting_model,0.85,0.805049,0.800567,0.877643,0.690601,0.035016,0.717274,0.717274
1,baseline_centralized_model,0.80,0.675319,0.614236,0.870902,0.594584,0.209842,0.598538,0.598538
2,central_logistic_baseline,0.90,0.576563,0.544385,0.858127,0.440240,0.164753,0.530251,0.530251
3,federated_performance_weighted_model,0.45,0.529811,0.427623,0.859092,0.683931,0.557874,0.484470,0.484470
4,federated_equal_weight_model,0.45,0.525734,0.417773,0.858132,0.669137,0.557946,0.478366,0.478366


Selected strategy: central_hist_gradient_boosting_model
